# CSPF — stage Q: quality control of a filled annotation sheet
Run this on every filled sheet (H4 blind sheet and H5 round-2 sheet), **before** I use the labels. For one or two annotators it checks:
1. **format**: no blank cells, only allowed lower-case values;
2. **attention**: accuracy on the 6 hidden gold rows (they have unambiguous answers);
3. **consistency**: no logical contradictions (meaning = yes with sentiment ≠ yes), no long runs of the same answer, comments used for doubts;
4. **plausibility**: label distribution against what round 1 looked like;
5. with two annotators: raw agreement and Cohen's κ per dimension (and per source/family), and the list of rows on which they disagree.
It prints PASS / WARN / FAIL per check and writes `Q_report.csv`. Output to send me: `CSPF_stageQ_bundle.zip`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"; D = f"{BASE}/results_FIXED/reviewer_H"; OUT = f"{BASE}/results_FIXED/reviewer_Q"
# which sheet to check: "H4" (blind sheet) or "H5" (round-2 sheet); file names of the FILLED sheets (annotator B is optional)
SHEET = "H5"
FILLED_A = f"{D}/H5_round2_sheet_FILLED_annotatorA.csv"; FILLED_B = f"{D}/H5_round2_sheet_FILLED_annotatorB.csv"          # set FILLED_B = None if there is only one annotator
KEY = f"{D}/H5_round2_KEY_do_not_share.csv"          # for H4: f"{D}/H4_blind_sheet_KEY_do_not_share.csv"
import os, subprocess, sys, warnings, zipfile, glob; warnings.filterwarnings("ignore"); os.makedirs(OUT, exist_ok=True)
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
ALLOWED = {"meaning": {"yes","partial","no"}, "sentiment": {"yes","unclear","no"}, "natural": {"yes","borderline","no"}}
def ensure(p):
    if p is None or os.path.exists(p): return p
    from google.colab import files; print("not found:", p, "-> please upload", os.path.basename(p)); up = files.upload(); n = list(up)[0]; os.replace(n, "/content/"+os.path.basename(p)); return "/content/"+os.path.basename(p)
FILLED_A, KEY = ensure(FILLED_A), ensure(KEY)
if FILLED_B is not None and not os.path.exists(FILLED_B): FILLED_B = None if input("annotator B file missing; continue with one annotator? (y/n) ").strip().lower() == "y" else ensure(FILLED_B)
key = pd.read_csv(KEY, encoding="utf-8-sig")
def load(p):
    a = pd.read_csv(p, encoding="utf-8-sig"); a.columns = [c.strip() for c in a.columns]
    ren = {c: c[:-2] for c in a.columns if c.endswith(("_a","_b")) and c[:-2] in ALLOWED}; a = a.rename(columns=ren)
    for c in ALLOWED: a[c] = a[c].astype(str).str.strip().str.lower().replace({"nan": ""})
    a["comment"] = a["comment"].fillna("").astype(str) if "comment" in a else (a["comment_a"].fillna("").astype(str) if "comment_a" in a else ""); return a
ANN = {"A": load(FILLED_A)}; 
if FILLED_B: ANN["B"] = load(FILLED_B)
REP = []
def rep(who, name, status, detail): REP.append(dict(annotator=who, check=name, status=status, detail=detail)); print(f"[{status:4s}] {who}: {name} — {detail}")

In [ ]:
for who, a in ANN.items():
    # 1 format
    n = len(a); rep(who, "rows", "PASS" if n == len(key) else "FAIL", f"{n} rows in the sheet, {len(key)} in the key")
    miss = {c: int((a[c] == "").sum()) for c in ALLOWED}; rep(who, "blank cells", "PASS" if sum(miss.values()) == 0 else "FAIL", str(miss))
    bad = {c: sorted(set(a[c]) - ALLOWED[c] - {""}) for c in ALLOWED}; rep(who, "invalid values", "PASS" if not any(bad.values()) else "FAIL", str({k: v for k, v in bad.items() if v}) or "none")
    d = a.merge(key, on="sample_id", how="left"); ANN[who] = d
    # 2 gold rows
    g = d[d.source == "GOLD"]
    for dim in ["meaning", "sentiment", "natural"]:
        exp = g["exp_"+dim].dropna(); ok = (g.loc[exp.index, dim] == exp).sum(); thr = len(exp) - (1 if dim == "meaning" else 0)
        rep(who, f"gold rows, {dim}", "PASS" if ok >= len(exp) - 1 else "FAIL", f"{ok}/{len(exp)} correct")
    # 3 consistency
    viol = int(((d.meaning == "yes") & (d.sentiment != "yes")).sum()); rep(who, "meaning=yes but sentiment!=yes", "PASS" if viol <= 3 else "WARN", f"{viol} rows (re-check them)")
    def longest(s):
        best = cur = 1
        for x, y in zip(s[:-1], s[1:]): cur = cur+1 if x == y else 1; best = max(best, cur)
        return best
    ordr = d.sort_values("sample_id"); runs = {c: longest(ordr[c].tolist()) for c in ALLOWED}; rep(who, "longest run of the same answer", "PASS" if max(runs.values()) <= 40 else "WARN", str(runs))
    cm = (d.comment.str.strip() != "").mean(); rep(who, "share of rows with a comment", "PASS" if cm >= 0.05 else "WARN", f"{cm:.0%} (very few comments can mean that rows were not read closely)")
    # 4 plausibility
    r = d[d.source != "GOLD"]; dist = {c: r[c].value_counts(normalize=True).round(2).to_dict() for c in ALLOWED}; print("   distribution (non-gold):", dist)
    sy = (r.sentiment == "yes").mean(); rep(who, "sentiment=yes share", "PASS" if sy >= 0.80 else "WARN", f"{sy:.0%} (round 1: 87–96%)")
    my = (r.meaning == "yes").mean(); rep(who, "meaning=yes share", "PASS" if 0.05 <= my <= 0.60 else "WARN", f"{my:.0%} (round 1: 17–39% over the whole sample)")
    allp = (r.meaning == "partial").mean(); rep(who, "meaning=partial share", "PASS" if allp <= 0.85 else "WARN", f"{allp:.0%} (very high values mean 'partial' is used as a default)")

In [ ]:
if "B" in ANN:
    a, b = ANN["A"].set_index("sample_id"), ANN["B"].set_index("sample_id"); ix = a.index.intersection(b.index); ix = [i for i in ix if a.loc[i, "source"] != "GOLD"]; rows = []
    for dim in ALLOWED:
        x, y = a.loc[ix, dim].values, b.loc[ix, dim].values; k = cohen_kappa_score(x, y) if len(set(x)|set(y)) > 1 else float("nan"); rows.append(dict(dimension=dim, rows=len(ix), raw_agreement=round((x == y).mean(), 3), kappa=round(k, 3)))
        rep("A vs B", f"{dim} agreement", "PASS" if (x == y).mean() >= (0.70 if dim == "meaning" else 0.60) else "WARN", f"raw {(x==y).mean():.0%}, kappa {k:.2f}" + (" (round 1: 77%, kappa 0.54)" if dim == "meaning" else ""))
    AG = pd.DataFrame(rows); print(AG.to_string(index=False)); AG.to_csv(f"{OUT}/Q_agreement.csv", index=False)
    grp = "perturbation_family" if "perturbation_family" in a else "source"
    fam = pd.DataFrame({"group": a.loc[ix, grp].fillna("?").values, "agree_meaning": (a.loc[ix, "meaning"].values == b.loc[ix, "meaning"].values)}).groupby("group").agg(rows=("agree_meaning", "size"), agree=("agree_meaning", "mean")).round(2); print(fam); fam.to_csv(f"{OUT}/Q_agreement_by_group.csv")
    dis = a.loc[ix, ["original_review", "perturbed_review", "meaning"]].join(b.loc[ix, ["meaning"]], rsuffix="_B").rename(columns={"meaning": "meaning_A"}); dis = dis[dis.meaning_A != dis.meaning_B]; dis.to_csv(f"{OUT}/Q_disagreements_to_discuss.csv"); print(len(dis), "rows with different meaning labels (written to Q_disagreements_to_discuss.csv)")
R = pd.DataFrame(REP); R.to_csv(f"{OUT}/Q_report.csv", index=False); print("\nsummary:", R.status.value_counts().to_dict())
for who, d in ANN.items(): d.drop(columns=[c for c in ["original_review", "perturbed_review"] if c in d]).to_csv(f"{OUT}/Q_labels_{who}.csv", index=False)
with zipfile.ZipFile(f"{OUT}/CSPF_stageQ_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv"): z.write(f, os.path.basename(f))
print("send me:", f"{OUT}/CSPF_stageQ_bundle.zip")
try:
    from google.colab import files; files.download(f"{OUT}/CSPF_stageQ_bundle.zip")
except Exception as e: print("download manually from Drive:", e)